In [2]:
import pandas as pd
# Fix column names if needed
df = pd.read_csv("./data/combined_dataset.csv")

# Rename if your review column is called 'drug_review'
if 'drug_review' in df.columns:
    df = df.rename(columns={'drug_review': 'review_text'})

# Keep only necessary columns
df = df[['review_text']].dropna(subset=['review_text'])

# Save back
df.to_csv("./data/combined_dataset.csv", index=False)
print("✅ Column names fixed")

✅ Column names fixed


In [12]:
# bert_classifier.py
import pandas as pd
import numpy as np
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
import joblib

# -----------------------------
# 1. Load and Prepare Data
# -----------------------------
# Update path as needed
df = pd.read_csv("src/data/combined_dataset.csv")  # Ensure this file exists

# Rename column if needed
if 'review' in df.columns:
    df = df.rename(columns={'review': 'review_text'})

# Drop rows with missing review text
df = df.dropna(subset=['review_text'])

# Define categories
categories = [
    "Positive_Experience",
    "Severe_Side_Effects",
    "Ineffective",
    "Dependency/Addiction",
    "Dosage_Issues",
    "Mixed_Feedback"
]

# Simple rule-based labeling
def classify_review_category(text):
    text_lower = text.lower()
    if any(word in text_lower for word in ['not work', 'no effect', 'didn’t help', 'useless', 'ineffective']):
        return "Ineffective"
    elif any(word in text_lower for word in ['addicted', 'withdrawal', 'dependence', 'craving']):
        return "Dependency/Addiction"
    elif any(word in text_lower for word in ['dosage', 'dose', 'too strong', 'too weak', 'missed dose']):
        return "Dosage_Issues"
    elif any(word in text_lower for word in ['nausea', 'dizzy', 'tired', 'insomnia', 'panic', 'rash', 'weight gain']):
        return "Severe_Side_Effects"
    elif any(word in text_lower for word in ['helped', 'works', 'great', 'relief', 'improved', 'stable', 'better', 'miracle']):
        return "Positive_Experience"
    else:
        return "Mixed_Feedback"

# Apply labeling
df['review_category'] = df['review_text'].apply(classify_review_category)

# Encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['review_category'])

# Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['review_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# -----------------------------
# 2. Tokenizer & Model
# -----------------------------
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
train_dataset = ReviewDataset(train_texts, train_labels, tokenizer)
val_dataset = ReviewDataset(val_texts, val_labels, tokenizer)

# Load model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label_encoder.classes_)
)

# -----------------------------
# 3. Compute Metrics (Fix for eval_accuracy)
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

# -----------------------------
# 4. Training Arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",           # Evaluates at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",  # Now works because we define it
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=10,
    report_to="none",                # Disable external logging (WandB, etc.)
    fp16=True,                       # Use mixed precision for speed (if GPU)
    remove_unused_columns=False      # Helps avoid some model errors
)

# -----------------------------
# 5. Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics   # Critical: This fixes the KeyError
)

# -----------------------------
# 6. Train the Model
# -----------------------------
print("🚀 Starting training...")
trainer.train()

# -----------------------------
# 7. Save Model & Label Encoder
# -----------------------------
print("✅ Training complete. Saving model...")
model.save_pretrained("./models/fine_tuned_distilbert_drug_reviews")
tokenizer.save_pretrained("./models/fine_tuned_distilbert_drug_reviews")
joblib.dump(label_encoder, "./models/label_encoder.pkl")

print("🎉 BERT model trained and saved successfully!")
print("📁 Model: ./models/fine_tuned_distilbert_drug_reviews")
print("📄 Label Encoder: ./models/label_encoder.pkl")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.059200,0.015722,0.997696
2,0.000600,0.011243,0.998574
3,0.000300,0.004325,0.999452


✅ Training complete. Saving model...
🎉 BERT model trained and saved successfully!
📁 Model: ./models/fine_tuned_distilbert_drug_reviews
📄 Label Encoder: ./models/label_encoder.pkl


In [ ]:
# src/test.py
from transformers import pipeline
import joblib

# Load model and tokenizer
classifier = pipeline(
    "text-classification",
    model=r"yogeshagowda/distilbert-drug-reviews",
    tokenizer=r"yogeshagowda/distilbert-drug-reviews"
)

# Load label encoder
label_encoder = joblib.load(r"./src/models/label_encoder.pkl")

def predict(text):
    result = classifier(text)[0]
    label_id = int(result['label'].split('_')[-1])
    predicted_class = label_encoder.classes_[label_id]
    confidence = result['score']
    return predicted_class, confidence

# Test
if __name__ == "__main__":
    test_review = "This drug helped me with depression but caused insomnia."
    category, conf = predict(test_review)
    print(f"Review: {test_review}")
    print(f"Predicted: {category} (Confidence: {conf:.2f})")

Device set to use cpu


Review: This drug helped me with depression but caused insomnia.
Predicted: Severe_Side_Effects (Confidence: 1.00)


c:\Users\yogesh.gowda\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
# src/review_category.py
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import joblib

# Load model and tokenizer
model_path = r"yogeshagowda/distilbert-drug-reviews"
tokenizer = DistilBertTokenizer.from_pretrained(model_path)
model = DistilBertForSequenceClassification.from_pretrained(model_path)
label_encoder = joblib.load(r"./src/models/label_encoder.pkl")

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # Set to evaluation mode

def predict_category(text):
    try:
        # Ensure input is string
        text = str(text)

        # Tokenize with truncation and padding
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,      # ← Critical: truncates to max length
            padding=True,         # pads to same length in batch
            max_length=512        # ← DistilBERT's max
        )

        # Move to device
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        # Inference
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Get predicted class
        predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        predicted_label = label_encoder.inverse_transform(predictions)[0]

        return predicted_label

    except Exception as e:
        print(f"Error predicting for text: {text[:100]}... → {e}")
        return "Mixed_Feedback"  # Fallback category

def add_categories_to_data(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    if 'drug_review' in df.columns:
        df.rename(columns={'drug_review': 'review_text'}, inplace=True)
    elif 'drug_review' not in df.columns and 'review_text' not in df.columns:
        raise ValueError("No review column found in CSV")

    # Apply prediction
    print("🧠 Applying BERT model to classify reviews...")
    df['predicted_category'] = df['review_text'].apply(predict_category)

    # Save result
    df.to_csv(output_csv, index=False)
    print(f"✅ Saved predictions to {output_csv}")
    return df

if __name__ == "__main__":
    # ✅ Update paths if needed
    input_file = "./data/combined_dataset.csv"
    output_file = "./data/drug_reviews_with_categories.csv"

    df = add_categories_to_data(input_file, output_file)
    print("\n📊 Sample predictions:")
    print(df[['review_text', 'predicted_category']].head())

🧠 Applying BERT model to classify reviews...


KeyboardInterrupt: 

In [14]:
# src/compute_trust_score.py
import pandas as pd
import numpy as np

# Category → Score mapping
CATEGORY_SCORES = {
    "Positive_Experience": +1.0,
    "Severe_Side_Effects": -1.0,
    "Ineffective": -0.8,
    "Dependency/Addiction": -0.9,
    "Dosage_Issues": -0.5,
    "Mixed_Feedback": 0.0
}

def compute_trust_score(input_csv, output_csv="./data/trust_scores.csv", min_reviews=1):
    """
    Compute Trust Score for each drug.
    If a drug has fewer than `min_reviews`, its trust score is 0.0
    """
    # Load data
    try:
        df = pd.read_csv(input_csv)
    except Exception as e:
        raise Exception(f"❌ Failed to read {input_csv}: {e}")

    # Rename if needed
    if 'drug_review' in df.columns:
        df.rename(columns={'drug_review': 'review_text'}, inplace=True)

    if 'drug_name' not in df.columns:
        raise ValueError("❌ 'drug_name' column not found in CSV")

    # Drop rows with missing predicted_category
    df = df.dropna(subset=['predicted_category'])

    # Map categories to scores
    df['category_score'] = df['predicted_category'].map(CATEGORY_SCORES)

    # Group by drug
    summary = df.groupby('drug_name').agg(
        raw_score=('category_score', 'mean'),
        total_reviews=('category_score', 'size'),
        positive_reviews=('predicted_category', lambda x: (x == "Positive_Experience").sum()),
        side_effects=('predicted_category', lambda x: (x == "Severe_Side_Effects").sum()),
        age_completion=('age', lambda x: x.notna().mean()),
        gender_completion=('gender', lambda x: x.notna().mean())
    ).reset_index()

    # Compute Trust Score: 0–1 scale
    summary['trust_score'] = (summary['raw_score'] + 1) / 2  # -1 → +1 becomes 0 → 1

    # 🔴 Set trust score to 0.0 if below minimum review threshold
    summary['trust_score'] = summary.apply(
        lambda row: 0.0 if row['total_reviews'] < min_reviews else row['trust_score'],
        axis=1
    )

    # Optional: Fill NaN trust scores (if any) with 0.0
    summary['trust_score'] = summary['trust_score'].fillna(0.0)

    # Sort by trust score
    summary = summary.sort_values('trust_score', ascending=False).reset_index(drop=True)

    # Save
    summary.to_csv(output_csv, index=False)
    print(f"✅ Trust scores computed and saved to {output_csv}")
    return summary

if __name__ == "__main__":
    result = compute_trust_score(
        input_csv="./data/drug_reviews_with_categories.csv",
        output_csv="./data/trust_scores.csv",
        min_reviews=1  # Require at least 1 real review
    )
    print("\n📊 Top Drugs by Trust Score:")
    print(result.head(10))

✅ Trust scores computed and saved to ./data/trust_scores.csv

📊 Top Drugs by Trust Score:
      drug_name  raw_score  total_reviews  positive_reviews  side_effects  \
0  zovia 1 / 50        1.0              1                 1             0   
1      disalcid        1.0              1                 1             0   
2      zyrtec-d        1.0              1                 1             0   
3       dolobid        1.0              1                 1             0   
4       dyazide        1.0              1                 1             0   
5   dynacirc cr        1.0              1                 1             0   
6    e-z-gas ii        1.0              1                 1             0   
7    e.e.s.-400        1.0              1                 1             0   
8    vicodin hp        1.0              1                 1             0   
9        vienva        1.0              1                 1             0   

   age_completion  gender_completion  trust_score  
0         